# Day 40 — Model tuning & pipelines (GridSearchCV/RandomizedSearchCV)
Objectives:
- Build a Pipeline.
- Perform parameter search.
- Discuss nested CV for unbiased estimates.


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.datasets import load_breast_cancer
X,y = load_breast_cancer(return_X_y=True)
pipe = Pipeline([('sc', StandardScaler()), ('svc', SVC())])
param_grid = {'svc__C':[0.1,1,10], 'svc__kernel':['linear','rbf']}
gs = GridSearchCV(pipe, param_grid, cv=5, scoring='roc_auc')
gs.fit(X,y)
gs.best_params_, gs.best_score_


## How to use this notebook

Select the `Python (ds60sqlpy)` kernel, start at the first cell, and
write each prediction before execution. Keep attempts in the
provided scratch cell or new cells. Restart the kernel and run from
the top before calling the work reproducible.

## Concept lab — hyperparameter search spaces, fit budgets, and nested evaluation

### Mental model

Hyperparameters configure how an estimator learns; a search procedure
chooses among candidate configurations using validation data. A grid is
a Cartesian product of listed values. Randomized search samples a fixed
number of configurations from distributions, which is often more
efficient when only a few dimensions matter.

Search results are themselves fitted to data. `best_score_` estimates
the best candidate on the inner validation folds and is optimistically
selected from many alternatives. A separate holdout or outer
cross-validation loop evaluates the entire selection procedure.

### Read the API before running it

- **`step__parameter`:** addresses nested pipeline parameters; inspect `get_params()` instead of guessing names.
- **`ParameterGrid(space)`:** enumerates the exact Cartesian search and makes the fit budget calculable.
- **`GridSearchCV(..., scoring=..., refit=...)`:** scores candidates in inner folds and refits the selected configuration on all supplied rows.

For every call, identify input data, learned state, returned value,
and a check that can fail. That habit prevents a successful cell
from being mistaken for a correct analysis.

### Focused example A — calculate the search budget before fitting

**Predict first:** write down the expected shape, type, ordering, or
direction of the result. Then run the next cell.

**Assumption:** Every listed combination is valid for the estimator and receives the same fold assignments.

In [ ]:
from sklearn.model_selection import ParameterGrid

space = {
    "model__C": [0.01, 0.1, 1.0, 10.0],
    "model__class_weight": [None, "balanced"],
    "model__solver": ["liblinear", "lbfgs"],
}
candidates = list(ParameterGrid(space))
folds = 5
expected_fits = len(candidates) * folds + 1  # final refit
print({"candidates": len(candidates), "expected_fits": expected_fits})
assert len(candidates) == 16

**Expected observation:** Three modest lists already create 16 candidates and 81 model fits with five folds and refit.

Do not force exact equality for estimates based on samples. Record
the seed, sample size, tolerance, and metric where they matter.

### Focused example B — make the refit metric explicit in multi-metric search

This example changes one important condition. Predict how and why
the result should differ from Example A.

**Assumption:** Macro F1 matches the decision and class-weighting priorities for this task.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_iris(return_X_y=True)
pipe = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=2_000)),
])
search = GridSearchCV(
    pipe,
    {"model__C": [0.1, 1.0, 10.0]},
    scoring={"accuracy": "accuracy", "f1_macro": "f1_macro"},
    refit="f1_macro",
    cv=StratifiedKFold(3, shuffle=True, random_state=4002),
).fit(X, y)
print(search.best_params_, search.best_score_)
assert search.refit == "f1_macro"

**Expected observation:** The selected estimator and `best_score_` are tied to `f1_macro`, not an implicit first metric.

### Debugging and practice ramp

**Common mistake:** Reporting `best_score_` as unbiased test performance after trying many candidates.

**Diagnostic:** Inspect `cv_results_`, candidate count, rank stability, train/validation gaps, failed fits, and whether search preprocessing lives inside the pipeline.

| Stage | Action | Evidence |
|---|---|---|
| Recall | Define hyperparameter search spaces, fit budgets, and nested evaluation in your own words and identify its input and output. | A definition that does not rely on the library name. |
| Predict | Predict the examples before execution, including shape and direction. | A written prediction and an explanation of any mismatch. |
| Implement | Recreate one example with a changed but valid input. | Code plus an assertion for the central invariant. |
| Debug | Trigger the named mistake or edge case intentionally. | The observed symptom and the smallest diagnostic that isolates it. |
| Transfer | Apply the idea to a different local dataset or decision. | A stated assumption, metric, and reason the method is suitable. |

**Stop condition:** Do not expand a search space without a fit budget, parameter rationale, and untouched evaluation plan.

Continue to the numbered practice only after you can explain both
examples without rereading their code.

## Learner exercises and progressive hints

1. Use `RandomizedSearchCV` with a wider parameter space.

**Verify:** Practice 1 — hyperparameter search spaces, fit budgets, and nested evaluation — declare parameter distributions, n_iter, scorer, CV splitter, and seed; print candidate count, expected fit count, best parameters, best CV score, failed-fit count, and one untouched-test metric.

2. Implement nested cross-validation and compare its result with the non-nested
   search score.

**Verify:** Practice 2 — hyperparameter search spaces, fit budgets, and nested evaluation — print every outer-fold score from a search fitted only inside that fold, its mean/std, and the optimistic non-nested best-CV score; assert outer validation indices never enter their inner search.

### Progressive hints

1. Sample `C` over orders of magnitude (for example with SciPy's `loguniform`)
   and set `random_state`. Start with about 20 iterations on a laptop.
2. Build separate inner and outer `StratifiedKFold` objects. Pass the entire
   search object—not its already-selected best estimator—to
   `cross_val_score` with the outer splitter.

### Additional mastery practice

Treat tuning as a finite experimental budget with a declared search space, selection metric, resampling design, and reproducible result table.

Predict or plan before you run code. Use the hint only after an honest
attempt, and record the evidence that would prove your result correct.

3. **Search-budget calculation:** For a grid with 5 values of C, 4 penalties, 3 class weights, and 5-fold CV, calculate candidate and fit counts. Then identify invalid solver/penalty combinations before running.
   **Progressive hint:** Cartesian-product candidates multiply; each candidate is fit once per fold, plus a possible final refit.

**Verify:** Search-budget calculation — show 5×4×3=60 raw combinations and 60×5=300 fits before filtering; list invalid solver/penalty pairs, print the valid candidate/fit count, and reconcile it with cv_results_ rows.

4. **Multi-metric selection:** Configure GridSearchCV to report ROC AUC, average precision, and balanced accuracy while refitting one declared metric. Explain why the refit choice belongs in the experiment plan.
   **Progressive hint:** Pass a scoring dictionary and set `refit` to a metric name. Selection changes when metrics rank candidates differently.

**Verify:** Multi-metric selection — configure all three scorers, print their mean/std/rank columns and the declared refit metric, and assert best_estimator_ corresponds to rank 1 for that metric rather than another scorer.

5. **Results-table diagnosis:** Turn `cv_results_` into a tidy table containing parameters, mean and standard deviation of train/validation scores, rank, and fit time. Flag overfit and unstable candidates.
   **Progressive hint:** Large train-validation gaps suggest overfit; large fold standard deviation suggests sensitivity. Sort by the declared rank, not by eye.

**Verify:** Results-table diagnosis — emit a tidy table with one row per candidate and explicit parameter, train mean/std, validation mean/std, rank, fit-time mean/std, and failure columns; flag candidates using declared train-validation-gap and variability rules.

6. **Reproducibility debugging:** A randomized search produces different winners on repeated runs. List every random source and parallelism setting to inspect, then design a deterministic comparison.
   **Progressive hint:** Seed the sampler, splitters, and estimator. Threaded numeric libraries and GPU algorithms can still introduce small nondeterminism.

**Verify:** Reproducibility debugging — run the search twice and match candidate order, scores, ranks, and winner after fixing data split, estimator, distribution sampler, CV, and library-thread seeds; record n_jobs and versions.

Before opening the reference solution, explain the relevant assumption,
failure mode, and validation check for every answer.

In [ ]:
# Expanded mastery lab scratch space
#
# Keep the official solution closed until you have attempted each task.
# Add small assertions, shape checks, or metric comparisons as evidence.

# Practice 3 — Search-budget calculation


# Practice 4 — Multi-metric selection


# Practice 5 — Results-table diagnosis


# Practice 6 — Reproducibility debugging
